[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/15_cross_validation/01_holdout_kfold_stratified.ipynb)

# Bài 01 — Hold-out, K-Fold và Stratified K-Fold

**🎯 Mục tiêu**
- Thấy **độ bất định** của một lần chia hold-out: cùng model, khác cách chia → điểm khác nhau đáng kể.
- Cài đặt **K-Fold** from scratch; báo cáo mean ± std; chọn $k$.
- **Stratified K-Fold** cho dữ liệu mất cân bằng; Repeated K-Fold.
- `cross_validate` (nhiều metric, train score) và `cross_val_predict` (dự đoán out-of-fold).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/15_cross_validation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (KFold, RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict, cross_val_score,
                                     cross_validate, train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
churn = make_customer_churn(n=1500, seed=2)
X = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float).to_numpy()
y = churn["churn"].to_numpy()
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
print(X.shape, "| tỉ lệ churn:", y.mean().round(3))

## 1. Một lần hold-out đáng tin đến đâu?

Chia train/test 80/20 với **100 random seed khác nhau**, cùng một model:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for test_size in [0.1, 0.2, 0.4]:
    aucs = []
    for seed in range(100):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)
        aucs.append(roc_auc_score(y_te, model.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]))
    ax.hist(aucs, bins=20, alpha=0.5, label=f"test {test_size:.0%}: {np.mean(aucs):.3f} ± {np.std(aucs):.3f} (min {min(aucs):.3f}, max {max(aucs):.3f})")
ax.set_xlabel("ROC-AUC trên tập test")
ax.legend(fontsize=8)
ax.set_title("Cùng model, 100 cách chia ngẫu nhiên khác nhau")
plt.show()

Điểm số dao động **vài điểm phần trăm** chỉ vì may rủi của việc chia — đủ để "model A tốt hơn model B" đổi chiều. Tập test nhỏ → dao động lớn; tập test lớn → ít dữ liệu train hơn. Cross-validation dùng **mọi** mẫu vừa để train vừa để đánh giá.

## 2. K-Fold from scratch

Chia dữ liệu (đã xáo trộn) thành $k$ phần; lần lượt mỗi phần làm validation, $k - 1$ phần còn lại làm train → $k$ điểm số → báo cáo **trung bình ± độ lệch chuẩn**.

In [ ]:
def kfold_indices(n, k=5, seed=0):
    idx = np.random.default_rng(seed).permutation(n)
    folds = np.array_split(idx, k)
    for i in range(k):
        val = folds[i]
        train = np.concatenate([folds[j] for j in range(k) if j != i])
        yield train, val


def cross_val_auc(model, X, y, splits):
    scores = []
    for train, val in splits:
        model.fit(X[train], y[train])
        scores.append(roc_auc_score(y[val], model.predict_proba(X[val])[:, 1]))
    return np.array(scores)


ours = cross_val_auc(model, X, y, kfold_indices(len(X), k=5))
sk = cross_val_score(model, X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="roc_auc")
print(f"scratch: {ours.round(3)} → {ours.mean():.4f} ± {ours.std():.4f}")
print(f"sklearn: {sk.round(3)} → {sk.mean():.4f} ± {sk.std():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.5))
for i, (train, val) in enumerate(kfold_indices(60, k=5, seed=0)):
    row = np.zeros(60)
    row[val] = 1
    ax.scatter(np.arange(60), np.full(60, i), c=row, cmap="coolwarm", marker="s", s=40)
ax.set_yticks(range(5), [f"fold {i + 1}" for i in range(5)])
ax.set_xlabel("chỉ số mẫu (60 mẫu minh hoạ)")
ax.set_title("K-Fold (k = 5): đỏ = validation, xanh = train — mỗi mẫu làm validation đúng 1 lần")
plt.show()

### Chọn k

| k | Train mỗi fold | Ước lượng | Chi phí |
|---|---|---|---|
| 2–3 | 50–67% dữ liệu | **bi quan** (model train trên ít dữ liệu hơn) | rẻ |
| **5–10** | 80–90% | cân bằng — lựa chọn mặc định | vừa |
| $n$ (LOO) | gần 100% | gần như không bi quan, nhưng variance cao | đắt ($n$ lần train) |

In [ ]:
for k in [2, 3, 5, 10, 20]:
    s = cross_val_score(model, X, y, cv=KFold(k, shuffle=True, random_state=0), scoring="roc_auc")
    print(f"k = {k:2d}: ROC-AUC = {s.mean():.4f} ± {s.std():.4f} (độ lệch chuẩn giữa các fold)")

## 3. Stratified K-Fold

Dữ liệu mất cân bằng + K-Fold thường → tỉ lệ positive ở mỗi fold dao động, thậm chí có fold **không có** positive (ROC-AUC không tính được). **Stratified** chia **từng lớp** riêng → mỗi fold giữ đúng tỉ lệ lớp.

In [ ]:
rng = np.random.default_rng(1)
y_rare = (rng.random(300) < 0.04).astype(int)                       # 4% positive: 15 mẫu
print("tổng số positive:", y_rare.sum())
for name, cv in [("KFold", KFold(5, shuffle=True, random_state=0)), ("StratifiedKFold", StratifiedKFold(5, shuffle=True, random_state=0))]:
    counts = [int(y_rare[val].sum()) for _, val in cv.split(np.zeros((300, 1)), y_rare)]
    print(f"{name:16s} số positive trong mỗi fold validation: {counts}")

Với phân loại, scikit-learn **mặc định** dùng `StratifiedKFold` khi bạn truyền `cv=5` cho classifier. Với hồi quy, có thể stratify theo **phân vị của y** (chia y thành bin rồi stratify).

## 4. Repeated (Stratified) K-Fold

Kết quả CV vẫn phụ thuộc cách xáo trộn. Lặp lại K-Fold nhiều lần với seed khác nhau → ước lượng ổn định hơn:

In [ ]:
single = [cross_val_score(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=s), scoring="roc_auc").mean() for s in range(10)]
repeated = cross_val_score(model, X, y, cv=RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0), scoring="roc_auc")
print(f"10 lần 5-fold (mỗi lần một seed): trung bình dao động từ {min(single):.4f} đến {max(single):.4f}")
print(f"Repeated 5×10: {repeated.mean():.4f} ± {repeated.std() / np.sqrt(len(repeated)):.4f} (sai số chuẩn — xấp xỉ, các fold không độc lập)")

## 5. `cross_validate` và `cross_val_predict`

In [ ]:
res = cross_validate(model, X, y, cv=5, scoring=["roc_auc", "average_precision", "accuracy"], return_train_score=True)
pd.DataFrame(res).drop(columns=["fit_time", "score_time"]).agg(["mean", "std"]).round(4)

- `train_*` ≫ `test_*` → overfit; cả hai thấp → underfit (chương 05).

`cross_val_predict` trả về dự đoán **out-of-fold** cho mọi mẫu — mỗi mẫu được dự đoán bởi model **không** thấy nó. Hữu ích cho: vẽ confusion matrix/PR curve trên toàn bộ dữ liệu, chọn ngưỡng, stacking (chương 24).

In [ ]:
oof = cross_val_predict(model, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
print(f"ROC-AUC trên dự đoán out-of-fold: {roc_auc_score(y, oof):.4f}")

> ⚠️ ROC-AUC trên **gộp** dự đoán out-of-fold không hoàn toàn bằng trung bình ROC-AUC từng fold (các fold dùng model khác nhau, thang điểm có thể lệch). Dùng `cross_val_score` để **ước lượng hiệu năng**; dùng `cross_val_predict` để **phân tích / chọn ngưỡng**.

## 🧠 Kiểm tra nhanh

1. Vì sao một lần hold-out không đủ để so sánh hai model có điểm chênh nhau 0.01?
2. 5-fold CV: mỗi mẫu được dùng làm validation bao nhiêu lần? Làm train bao nhiêu lần?
3. Khi nào bắt buộc dùng Stratified K-Fold?
4. Vì sao không nên báo cáo ROC-AUC trên dự đoán out-of-fold như ước lượng chính thức?

## 📌 Tóm tắt

- Hold-out có variance lớn; K-Fold tận dụng mọi mẫu, báo cáo mean ± std.
- $k$ = 5–10 là mặc định; Stratified cho phân loại (đặc biệt mất cân bằng); Repeated để ổn định.
- `cross_validate` cho nhiều metric + train score; `cross_val_predict` cho phân tích out-of-fold.

## 📚 Tài liệu tham khảo

- Kohavi (1995) — *A Study of Cross-Validation and Bootstrap for Accuracy Estimation and Model Selection*
- ESL §7.10
- scikit-learn User Guide — Cross-validation: evaluating estimator performance